# q6: time patterns
when do people buy? daily trend, day of week, and hour of day (sql/08_time_patterns.sql).
all times are utc (matches how the dataset defines days, see q6c).
day of week and hour use the clean window only (nov 12 – jan 25): transaction ids are missing nov 1–10 and jan 26–31 (q6b).
hour of day is flat, likely obfuscated timestamps, so it is not treated as a real pattern.


In [1]:
import warnings
warnings.filterwarnings("ignore", message="BigQuery Storage module not found")

from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project="ga4-funnel-analysis-511719")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)


In [2]:
sql = """
with sessions as (
    select
        date(timestamp_micros(session_start_time)) as session_date,
        purchased,
        purchased_any,
        revenue
    from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
)
select
    session_date,
    session_date between date '2020-11-12' and date '2021-01-25' as in_clean_window,
    count(*) as sessions,
    countif(purchased) as purchases,
    countif(purchased_any) as purchased_any,
    sum(revenue) as revenue
from sessions
group by session_date, in_clean_window
order by session_date
"""
daily = client.query(sql).to_dataframe()
daily

,session_date,in_clean_window,sessions,purchases,purchased_any,revenue
0,2020-11-01,False,2625,0,13,0.0
1,2020-11-02,False,3726,0,40,0.0
2,2020-11-03,False,5347,0,48,0.0
3,2020-11-04,False,4325,0,28,0.0
4,2020-11-05,False,3840,0,26,0.0
...,...,...,...,...,...,...
87,2021-01-27,False,4599,12,51,457.0
88,2021-01-28,False,4216,5,55,128.0
89,2021-01-29,False,3791,6,53,190.0
90,2021-01-30,False,2900,5,26,159.0


In [3]:
daily["conversion_rate"] = (daily["purchases"] / daily["sessions"]).round(4)
daily["revenue_per_session"] = (daily["revenue"] / daily["sessions"]).round(2)
clean=daily[daily["in_clean_window"]]
print(len(daily), daily["sessions"].sum(), daily["purchases"].sum(), daily["revenue"].sum())
print(len(clean), clean["sessions"].sum(), clean["purchases"].sum(), clean["revenue"].sum())
daily

92 360129 4044 307640.0
75 299173 3968 304028.0


,session_date,in_clean_window,sessions,purchases,purchased_any,revenue,conversion_rate,revenue_per_session
0,2020-11-01,False,2625,0,13,0.0,0.0,0.0
1,2020-11-02,False,3726,0,40,0.0,0.0,0.0
2,2020-11-03,False,5347,0,48,0.0,0.0,0.0
3,2020-11-04,False,4325,0,28,0.0,0.0,0.0
4,2020-11-05,False,3840,0,26,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
87,2021-01-27,False,4599,12,51,457.0,0.0026,0.1
88,2021-01-28,False,4216,5,55,128.0,0.0012,0.03
89,2021-01-29,False,3791,6,53,190.0,0.0016,0.05
90,2021-01-30,False,2900,5,26,159.0,0.0017,0.05


In [4]:
daily.to_csv(output_dir / "q6_daily.csv", index=False)

In [5]:
sql = """
with clean_sessions as (
    select
        date(timestamp_micros(session_start_time)) as session_date,
        purchased,
        revenue
    from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
    where date(timestamp_micros(session_start_time)) between date '2020-11-12' and date '2021-01-25'
)

select
    extract(dayofweek from session_date) as day_number,
    format_date("%A", session_date) as day_of_week,
    count(distinct session_date) as days,
    count(*) as sessions,
    countif(purchased) as purchases,
    sum(revenue) as revenue
from clean_sessions
group by day_number, day_of_week
order by day_number
"""

weekdays = client.query(sql).to_dataframe()
weekdays["sessions_per_day"] = (weekdays["sessions"] / weekdays["days"]).round(0)
weekdays["conversion_rate"] = (weekdays["purchases"] / weekdays["sessions"]).round(4)
weekdays["revenue_per_session"] = (weekdays["revenue"] / weekdays["sessions"]).round(2)
print(weekdays["days"].sum(), weekdays["purchases"].sum())
weekdays

75 3968


,day_number,day_of_week,days,sessions,purchases,revenue,sessions_per_day,conversion_rate,revenue_per_session
0,1,Sunday,11,32887,307,21460.0,2990.0,0.0093,0.65
1,2,Monday,11,44822,608,50271.0,4075.0,0.0136,1.12
2,3,Tuesday,10,47437,673,49836.0,4744.0,0.0142,1.05
3,4,Wednesday,10,47367,661,54465.0,4737.0,0.014,1.15
4,5,Thursday,11,47706,602,45644.0,4337.0,0.0126,0.96
5,6,Friday,11,43826,680,49958.0,3984.0,0.0155,1.14
6,7,Saturday,11,35128,437,32394.0,3193.0,0.0124,0.92


In [6]:
weekdays.to_csv(output_dir / "q6_day_of_week.csv", index=False)

In [7]:
sql = """
with clean_sessions as (
    select
        extract(hour from timestamp_micros(session_start_time)) as session_hour,
        purchased,
        revenue
    from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
    where date(timestamp_micros(session_start_time)) between date '2020-11-12' and date '2021-01-25'
)

select
    session_hour,
    count(*) as sessions,
    countif(purchased) as purchases,
    sum(revenue) as revenue
from clean_sessions
group by session_hour
order by session_hour
"""

hours = client.query(sql).to_dataframe()
hours["conversion_rate"] = (hours["purchases"] / hours["sessions"]).round(4)
hours["revenue_per_session"] = (hours["revenue"] / hours["sessions"]).round(2)
print(len(hours), hours["purchases"].sum())
hours

24 3968


,session_hour,sessions,purchases,revenue,conversion_rate,revenue_per_session
0,0,12504,152,13833.0,0.0122,1.11
1,1,12520,153,12619.0,0.0122,1.01
2,2,12348,162,12573.0,0.0131,1.02
3,3,12475,173,13633.0,0.0139,1.09
4,4,12389,161,10690.0,0.013,0.86
5,5,12705,187,14318.0,0.0147,1.13
6,6,12493,159,12228.0,0.0127,0.98
7,7,12392,176,12661.0,0.0142,1.02
8,8,12467,158,11431.0,0.0127,0.92
9,9,12674,177,13058.0,0.014,1.03


In [8]:
hours.to_csv(output_dir / "q6_hour_of_day.csv", index=False)